# Check Detection Against Raw Frame

Load a saved run directory, select a detection row from `detections.csv`, pull the corresponding raw video frame, and draw the detection overlay for inspection.

Notes:
- New single-camera runs record `raw.mp4` on the detection-aligned branch, so `frame` should match the saved video directly.
- Older runs and multi-camera runs may still rely on `raw_frame_num` when present.
- For dual-camera runs, it automatically selects `raw.mp4` for `cam0` and `raw_cam1.mp4` for `cam1`.


In [4]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib.patches import Rectangle

plt.rcParams["figure.figsize"] = (14, 8)

# Point this at a completed run directory.
RUN_DIR = Path("/home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/")

# Manual frame lookup is the default workflow.
# Set MANUAL_FRAME_NUM to a raw video frame number to inspect that frame directly.
# Leave it as None only if you want to fall back to selecting a detection row by ROW_INDEX.
MANUAL_FRAME_NUM = 11
TARGET_STREAM_ID = 0
ROW_INDEX = None

# If True, draw every detection that matches the selected frame + stream.
# If False, draw only the selected detection row.
SHOW_ALL_ON_FRAME = True

# How many frames to move when clicking Previous / Next.
FRAME_STEP = 1

# Video frame indexing in OpenCV is zero-based. If your stored frame number behaves
# like one-based indexing, set this to 1.
FRAME_INDEX_SHIFT = 0

# For validation, prefer sequential decode because random seek in H264/MP4 is not
# always frame-exact in OpenCV.
USE_SEQUENTIAL_DECODE = True

# Only draw keypoints with score >= this threshold.
KEYPOINT_SCORE_THRESHOLD = 0.25

In [5]:
def load_detections(run_dir: Path) -> list[dict[str, str]]:
    det_path = run_dir / "detections.csv"
    if not det_path.exists():
        raise FileNotFoundError(f"Missing detections.csv: {det_path}")
    with det_path.open(newline="") as fh:
        return list(csv.DictReader(fh))


def detection_video_path(run_dir: Path, row: dict[str, str]) -> Path:
    source = (row.get("source") or "").strip().lower()
    stream_id = int((row.get("stream_id") or 0))
    if source == "cam1" or stream_id == 1:
        path = run_dir / "raw_cam1.mp4"
    else:
        path = run_dir / "raw.mp4"
    if not path.exists():
        raise FileNotFoundError(f"Missing raw video file: {path}")
    return path


def frame_number_for_row(row: dict[str, str]) -> int:
    raw_frame = (row.get("raw_frame_num") or "").strip()
    if raw_frame:
        return int(raw_frame)
    return int(row["frame"])


def load_frame(video_path: Path, frame_num: int, *, shift: int = 0, sequential: bool = True):
    frame_idx = int(frame_num) - int(shift)
    if frame_idx < 0:
        raise ValueError(f"Computed negative frame index: {frame_idx}")
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    try:
        if sequential:
            ok = False
            frame_bgr = None
            current_idx = -1
            while current_idx < frame_idx:
                ok, frame_bgr = cap.read()
                if not ok or frame_bgr is None:
                    raise RuntimeError(f"Could not read frame {frame_idx} from {video_path}")
                current_idx += 1
        else:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, frame_bgr = cap.read()
            if not ok or frame_bgr is None:
                raise RuntimeError(f"Could not read frame {frame_idx} from {video_path}")
    finally:
        cap.release()
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return frame_rgb, frame_idx


def matching_rows(rows: list[dict[str, str]], selected: dict[str, str]) -> list[dict[str, str]]:
    raw_frame_num = str(frame_number_for_row(selected))
    stream_id = str(int(selected.get("stream_id") or 0))
    matches = []
    for row in rows:
        row_stream = str(int(row.get("stream_id") or 0))
        row_frame = str(frame_number_for_row(row))
        if row_stream == stream_id and row_frame == raw_frame_num:
            matches.append(row)
    return matches


def select_row(rows: list[dict[str, str]]) -> dict[str, str]:
    if MANUAL_FRAME_NUM is not None:
        target_stream = None if TARGET_STREAM_ID is None else int(TARGET_STREAM_ID)
        for row in rows:
            row_frame = frame_number_for_row(row)
            row_stream = int(row.get("stream_id") or 0)
            if row_frame != int(MANUAL_FRAME_NUM):
                continue
            if target_stream is not None and row_stream != target_stream:
                continue
            return row
        raise LookupError(
            f"No detection found for raw_frame_num={MANUAL_FRAME_NUM} stream_id={TARGET_STREAM_ID}"
        )
    if ROW_INDEX is None:
        raise ValueError("Set MANUAL_FRAME_NUM or provide a ROW_INDEX.")
    if ROW_INDEX < 0 or ROW_INDEX >= len(rows):
        raise IndexError(f"ROW_INDEX {ROW_INDEX} out of range for {len(rows)} detections")
    return rows[ROW_INDEX]


def _parse_keypoints(row: dict[str, str]) -> list[list[float]]:
    raw = (row.get("kpt_values_json") or "").strip()
    if not raw:
        return []
    try:
        parsed = json.loads(raw)
    except Exception:
        return []
    if not isinstance(parsed, list):
        return []
    points: list[list[float]] = []
    for item in parsed:
        if not isinstance(item, list) or len(item) < 3:
            continue
        try:
            points.append([float(item[0]), float(item[1]), float(item[2])])
        except Exception:
            continue
    return points


def draw_rows(ax, frame_rgb, rows_to_draw: list[dict[str, str]], title: str) -> None:
    ax.imshow(frame_rgb)
    ax.set_title(title)
    ax.axis("off")
    colors = ["lime", "cyan", "yellow", "magenta", "orange", "red"]
    for idx, row in enumerate(rows_to_draw):
        x = float(row["x"])
        y = float(row["y"])
        w = float(row["w"])
        h = float(row["h"])
        label = row.get("class_label") or f"class_{row.get('class_id', '?')}"
        conf = row.get("conf", "")
        color = colors[idx % len(colors)]
        ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=2, edgecolor=color))
        ax.text(
            x,
            max(0, y - 6),
            f"{idx}: {label} {conf}",
            color=color,
            fontsize=10,
            bbox={"facecolor": "black", "alpha": 0.6, "pad": 2},
        )
        for kp_idx, (kx, ky, ks) in enumerate(_parse_keypoints(row)):
            if ks < KEYPOINT_SCORE_THRESHOLD:
                continue
            ax.scatter([kx], [ky], s=26, c=color, marker="o", edgecolors="black", linewidths=0.4)
            ax.text(
                kx + 3,
                ky + 3,
                str(kp_idx),
                color="white",
                fontsize=7,
                bbox={"facecolor": color, "alpha": 0.55, "pad": 1},
            )


In [ ]:
rows = load_detections(RUN_DIR)
selected = select_row(rows)
video_path = detection_video_path(RUN_DIR, selected)

from io import BytesIO
from IPython.display import display, clear_output
from PIL import Image

selected_stream_id = int(selected.get("stream_id") or 0)
start_frame_num = frame_number_for_row(selected)

def rows_for_frame_num(rows: list[dict[str, str]], frame_num: int, stream_id: int) -> list[dict[str, str]]:
    matches = []
    for row in rows:
        row_stream = int(row.get("stream_id") or 0)
        row_frame = frame_number_for_row(row)
        if row_stream == stream_id and row_frame == int(frame_num):
            matches.append(row)
    return matches

def render_frame(frame_num: int) -> None:
    frame_rgb, video_frame_idx = load_frame(
        video_path,
        frame_num,
        shift=FRAME_INDEX_SHIFT,
        sequential=USE_SEQUENTIAL_DECODE,
    )
    frame_rows = rows_for_frame_num(rows, frame_num, selected_stream_id)
    rows_to_draw = frame_rows if SHOW_ALL_ON_FRAME else frame_rows[:1]
    fig, axes = plt.subplots(1, 2)
    axes[0].imshow(frame_rgb)
    axes[0].set_title(
        f"Raw frame only\nvideo={video_path.name} raw_frame_num={frame_num} video_idx={video_frame_idx}"
    )
    axes[0].axis("off")
    title = f"Overlay check\nsource=cam{selected_stream_id} raw_frame={frame_num} detections_on_frame={len(frame_rows)}"
    draw_rows(axes[1], frame_rgb, rows_to_draw, title)
    plt.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    with output:
        clear_output(wait=True)
        display(Image.open(buf))
        print(f"Frame {frame_num} on stream {selected_stream_id}")
        if frame_rows:
            print("Detection rows on this frame:")
            for idx, row in enumerate(frame_rows):
                print(idx, {k: row[k] for k in ['frame', 'raw_frame_num', 'stream_id', 'source', 'class_label', 'conf'] if k in row})
        else:
            print("No detection rows on this frame.")

state = {"frame_num": int(start_frame_num)}
frame_input = widgets.IntText(value=int(start_frame_num), description="Frame:")
prev_btn = widgets.Button(description="Previous")
next_btn = widgets.Button(description="Next")
go_btn = widgets.Button(description="Go")
output = widgets.Output()

def _set_frame(frame_num: int) -> None:
    state["frame_num"] = max(0, int(frame_num))
    frame_input.value = state["frame_num"]
    render_frame(state["frame_num"])

def _prev(_):
    _set_frame(state["frame_num"] - FRAME_STEP)

def _next(_):
    _set_frame(state["frame_num"] + FRAME_STEP)

def _go(_):
    _set_frame(frame_input.value)

prev_btn.on_click(_prev)
next_btn.on_click(_next)
go_btn.on_click(_go)

display(widgets.HBox([prev_btn, next_btn, frame_input, go_btn]))
display(output)
render_frame(state["frame_num"])

print("Selected anchor detection row:")
selected


Output()

Selected anchor detection row:


{'frame': '11',
 'raw_frame_num': '11',
 'ts_us': '585013',
 'stream_id': '0',
 'source': 'cam0',
 'obj_id': '18446744073709551615',
 'class_id': '0',
 'class_label': 'mouse',
 'conf': '0.904785',
 'x': '568.125',
 'y': '329.625',
 'w': '174.375',
 'h': '172.125',
 'pose_schema': 'mouse_v1',
 'kpt_count': '19',
 'kpt_names_json': '["kp0","kp1","kp2","kp3","kp4","kp5","kp6","kp7","kp8","kp9","kp10","kp11","kp12","kp13","kp14","kp15","kp16","kp17","kp18"]',
 'kpt_values_json': '[[640.688,343.125,0.981],[629.438,379.125,0.997],[610.312,385.312,0.995],[617.062,389.25,0.992],[662.062,442.688,0.997],[730.125,483.188,0.958],[662.062,381.656,0.0],[649.688,406.125,0.0],[659.812,363.938,0.0],[680.625,371.25,0.0],[669.375,449.438,0.0],[634.5,349.875,0.0],[617.062,322.875,0.0],[623.812,472.5,0.0],[574.594,500.625,0.0],[640.125,414.562,0.997],[691.312,465.188,0.976],[658.125,487.688,0.988],[666.0,488.25,0.973]]'}

RuntimeError: Could not read frame 286 from /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/raw.mp4

RuntimeError: Could not read frame 287 from /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/raw.mp4

...

RuntimeError: Could not read frame 290 from /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/raw.mp4

RuntimeError: Could not read frame 291 from /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/raw.mp4

RuntimeError: Could not read frame 292 from /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/raw.mp4

RuntimeError: Could not read frame 293 from /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/raw.mp4

RuntimeError: Could not read frame 294 from /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/raw.mp4